# 02 - ColPali Indexing & Retrieval (Kaggle / Colab GPU)

ColPali (PaliGemma-3B base, multi-vector) is too heavy to run on an 8GB M1,
so its index is built **here on a GPU notebook**. The ColPali-vs-BiomedCLIP
comparison is produced at the end; the ColPali index can also be downloaded
back into `data/index/colpali/` locally.

**Runtime:** enable a GPU - Kaggle: `Settings -> Accelerator -> GPU T4 x2`;
Colab: `Runtime -> Change runtime type -> T4 GPU`.

Steps: install deps -> clone the repo -> attach the MIMIC-CXR dataset ->
build the ColPali index -> compare ColPali vs BiomedCLIP -> download.

In [1]:
# 1. Install dependencies (ColPali is NOT in requirements.txt - it is Colab-only).
!pip install -q "colpali-engine>=0.3.0,<0.4.0" open_clip_torch faiss-cpu \
    transformers accelerate pyyaml python-dotenv pandas pillow tqdm sacrebleu rouge-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 33.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 77.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 109.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 112.7 MB/s eta 0:00:0000:01


In [ ]:
# 2. Clone the project repo and enter it.
import os, sys, pathlib

REPO_URL = "https://github.com/manohosny/DSAI-413-A2.git"
REPO_DIR = "DSAI-413-A2"

if pathlib.Path.cwd().name != REPO_DIR:
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL}
    os.chdir(REPO_DIR)
sys.path.insert(0, str(pathlib.Path.cwd() / "src"))
print("cwd:", pathlib.Path.cwd())

## 3 - Data

ColPali indexes the **report text** (rendered as page images), so only the
reports CSV is needed here - not the X-ray pixels.

In the Kaggle editor: **Add Input** (right sidebar) -> search
`simhadrisadaram/mimic-cxr-dataset` -> Add. It mounts at
`/kaggle/input/mimic-cxr-dataset/`. The next cell links the **validate** CSV
into `data/raw/` - the same split the local QA dataset was built from, so the
`study_id`s line up with `data/qa/qa_dataset.json` (which ships with the repo).

In [ ]:
# Link the MIMIC-CXR validate CSV into data/raw/ so the loader finds it.
import os

KAGGLE_INPUT = "/kaggle/input/mimic-cxr-dataset"
os.makedirs("data/raw", exist_ok=True)

csv_src = os.path.join(KAGGLE_INPUT, "mimic_cxr_aug_validate.csv")
csv_dst = "data/raw/mimic_cxr_aug_validate.csv"
if not os.path.exists(csv_dst):
    if not os.path.exists(csv_src):
        raise FileNotFoundError(
            f"{csv_src} missing - use 'Add Input' to attach the Kaggle "
            "dataset 'simhadrisadaram/mimic-cxr-dataset'."
        )
    os.symlink(csv_src, csv_dst)

from cxr.config import CONFIG
from cxr.data.loader import load_records

records = load_records(limit=300)   # matches the local BiomedCLIP index size
print(f"Loaded {len(records)} reports.")

## 4 - Build the ColPali index (renders each report as a page image)

In [ ]:
# ColPali's base (PaliGemma-3B) is gated - authenticate with an HF token.
# Add HF_TOKEN as a Kaggle Secret (Add-ons -> Secrets), or paste when prompted.
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None
if hf_token:
    login(hf_token)
    print("HF authenticated.")
else:
    print("No HF_TOKEN found - ColPali download may fail if its base is gated.")

# Build the ColPali index (renders each report as a page image).
from cxr.models.retrievers import build_retriever

colpali = build_retriever("colpali")
colpali.build_index(records)        # writes data/index/colpali/
print("ColPali index built.")

## 5 - Compare ColPali vs BiomedCLIP on the QA eval set

In [ ]:
import random
import pandas as pd
from cxr.data.qa_builder import load_qa_dataset
from cxr.evaluation.retrieval_eval import evaluate_retriever

# The BiomedCLIP index is gitignored, so build it here over the SAME 300
# reports - an apples-to-apples comparison with ColPali.
bm = build_retriever("biomedclip")
try:
    bm.load_index()
except FileNotFoundError:
    bm.build_index(records)

qa = load_qa_dataset()
qa_sample = random.Random(CONFIG.seed).sample(qa, min(CONFIG.evaluation.sample_size, len(qa)))

rows = []
for name in ["biomedclip", "colpali"]:
    try:
        r = build_retriever(name)
        r.load_index()
        rows.append(evaluate_retriever(r, qa_sample))
        print(f"{name}: scored {len(qa_sample)} queries")
    except Exception as exc:
        print(f"{name} skipped: {exc}")
pd.DataFrame(rows)

## 6 - Download the ColPali index back to your machine

In [ ]:
import shutil
shutil.make_archive('colpali_index', 'zip', 'data/index/colpali')
try:
    from google.colab import files
    files.download('colpali_index.zip')   # unzip into data/index/colpali/ locally
except ImportError:
    print('Not on Colab - archive saved as colpali_index.zip')